In [ ]:
# Step 1.1: Imports & basic config

import os
import random

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

# (Word2Vec etc.)
# from gensim.models import Word2Vec

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Step 1.2: Load cleaned dataset


DATA_PATH = "/kaggle/input/dataset/sentiment_mental_health_clean.csv"

df = pd.read_csv(DATA_PATH)

print("Data shape:", df.shape)
df.head()


In [ ]:
# Step 1.3: Basic sanity checks

print("Columns:", df.columns.tolist(), "\n")

# Quick look at label distribution
print("Label distribution (status):")
print(df["status"].value_counts(), "\n")

In [ ]:
# Step 1.4 (optional): Quick text length summary WITHOUT modifying df

text_lengths = df["statement"].astype(str).str.split().str.len()

print(text_lengths.describe())

In [ ]:
# Step 2.1: Encode labels (status -> integer IDs)

# Get unique labels in a stable order (sorted for reproducibility)
unique_labels = sorted(df["status"].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Label mapping (status -> id):")
for k, v in label2id.items():
    print(f"{k:20s} -> {v}")

# Add an integer label column
df["label_id"] = df["status"].map(label2id)

df[["statement", "status", "label_id"]].head()


In [ ]:
# Step 2.2: Stratified train/val/test split

TEST_SIZE = 0.15   # 15% test
VAL_SIZE  = 0.15   # 15% validation (of the *total* dataset)

# First split off the test set from the full data
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["label_id"]
)

# Now split train_val into train and val
# desired val fraction relative to train_val:
val_relative = VAL_SIZE / (1.0 - TEST_SIZE)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_relative,
    random_state=SEED,
    stratify=train_val_df["label_id"]
)

print("Total:", df.shape)
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)


In [ ]:
# Step 2.3: Reset index for cleanliness and quick peek

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Train sample:")
display(train_df.head())

print("Validation sample:")
display(val_df.head())

print("Test sample:")
display(test_df.head())

In [ ]:
# Step 2.4: Visualize label distributions with custom colors

import matplotlib.pyplot as plt
import seaborn as sns

# Custom color mapping
status_colors = {
    "Normal":               "#8FB9A8", 
    "Depression":           "#F2C57C",  
    "Suicidal":             "#F28F6B", 
    "Anxiety":              "#FFB4A2",  
    "Bipolar":              "#A5B5D9", 
    "Stress":               "#E4B7E5",  
    "Personality disorder": "#A4A3F1"
}

palette = [status_colors[label] for label in unique_labels]

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
sns.countplot(
    data=train_df,
    x="status",
    order=unique_labels,
    palette=palette
)
plt.title("Train label distribution")
plt.xticks(rotation=45, ha="right")

plt.subplot(1, 3, 2)
sns.countplot(
    data=val_df,
    x="status",
    order=unique_labels,
    palette=palette
)
plt.title("Validation label distribution")
plt.xticks(rotation=45, ha="right")

plt.subplot(1, 3, 3)
sns.countplot(
    data=test_df,
    x="status",
    order=unique_labels,
    palette=palette
)
plt.title("Test label distribution")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()


In [ ]:
# Step 3.1: Tokenizer and tokenized text for train/val/test

def tokenize(text):
    """
    tokenizer: lowercases (already should be),
    splits on whitespace, and strips extra spaces.
    """
    if not isinstance(text, str):
        text = str(text)
    return text.strip().split()

# Apply tokenizer to each split
train_tokens = train_df["statement"].astype(str).apply(tokenize)
val_tokens   = val_df["statement"].astype(str).apply(tokenize)
test_tokens  = test_df["statement"].astype(str).apply(tokenize)

# Quick check
print("Example tokenized train statement:")
print(train_df["statement"].iloc[1])
print(train_tokens.iloc[1])


In [ ]:
# Step 3.3: Decide on MAX_LEN for padding / truncation

MAX_LEN = 200

In [ ]:
# Step 3.4: Build vocabulary from training tokens only

from collections import Counter

# Count word frequencies in training data
word_counter = Counter()
for tokens in train_tokens:
    word_counter.update(tokens)

print(f"Total unique tokens in train: {len(word_counter)}")

# Special tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

# Start vocab with special tokens
word2idx = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

# Add remaining words
for word in word_counter.keys():
    if word not in word2idx:
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

vocab_size = len(word2idx)
print(f"Final vocab size (including PAD/UNK): {vocab_size}")


In [ ]:
# Step 3.5 (optional): Visualize top frequent words in train

import seaborn as sns
import matplotlib.pyplot as plt

TOP_N = 25
most_common_words = word_counter.most_common(TOP_N)
words, freqs = zip(*most_common_words)

plt.figure(figsize=(10, 5))
sns.barplot(x=np.array(words), y=list(freqs))
plt.title(f"Top {TOP_N} most frequent words (train)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Step 3.6: Convert tokens to padded integer sequences

import numpy as np

def encode_and_pad(tokens, word2idx, max_len):
    """
    Convert a list of tokens to a list of indices and pad/truncate to max_len.
    """
    ids = []
    for tok in tokens:
        ids.append(word2idx.get(tok, word2idx[UNK_TOKEN]))
    # Truncate
    ids = ids[:max_len]
    # Pad
    if len(ids) < max_len:
        ids = ids + [word2idx[PAD_TOKEN]] * (max_len - len(ids))
    return ids

# Apply to each split
X_train = np.array([encode_and_pad(toks, word2idx, MAX_LEN) for toks in train_tokens])
X_val   = np.array([encode_and_pad(toks, word2idx, MAX_LEN) for toks in val_tokens])
X_test  = np.array([encode_and_pad(toks, word2idx, MAX_LEN) for toks in test_tokens])

y_train = train_df["label_id"].values
y_val   = val_df["label_id"].values
y_test  = test_df["label_id"].values

print("X_train shape:", X_train.shape)
print("X_val shape  :", X_val.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_val shape  :", y_val.shape)
print("y_test shape :", y_test.shape)


In [ ]:
# Step 4.1: Import Word2Vec and prepare corpus (list-of-lists)

from gensim.models import Word2Vec

# Corpus for Word2Vec -> ONLY train set (to avoid leakage)
corpus = train_tokens.tolist()

print("Number of training sentences:", len(corpus))
print("Example tokenized sentence:", corpus[0][:20])


In [ ]:
# Step 4.2: Train Word2Vec model

EMBED_DIM = 300          # You can use 100 or 300 (100 is faster, 300 better)
WINDOW_SIZE = 5          # context window
MIN_COUNT = 2            # ignore extremely rare words
SG = 1                   # 1 = Skip-gram, 0 = CBOW

w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=EMBED_DIM,
    window=WINDOW_SIZE,
    min_count=MIN_COUNT,
    workers=4,       # number of CPU cores
    sg=SG
)

print("Word2Vec training complete.")
print("Vocabulary size inside Word2Vec:", len(w2v_model.wv))


In [ ]:
# Step 4.3: Build embedding matrix

import numpy as np

vocab_size = len(word2idx)

# Initialize embedding matrix with random vectors
embedding_matrix = np.random.uniform(
    low=-0.05, high=0.05, size=(vocab_size, EMBED_DIM)
).astype(np.float32)

# Replace vectors for words present in Word2Vec
found = 0
for word, idx in word2idx.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        found += 1

print(f"Found pretrained vectors for {found}/{vocab_size} words.")


In [ ]:
# Step 4.4: Convert embedding matrix to a PyTorch tensor

import torch

embedding_matrix_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)

print("Embedding matrix tensor shape:", embedding_matrix_tensor.shape)

In [ ]:
# Step 4.5 (optional): Try some similarity checks

test_words = ["anxiety", "depression", "sad", "suicidal", "worried"]

for w in test_words:
    if w in w2v_model.wv:
        print(f"\nTop words similar to '{w}':")
        for sim, score in w2v_model.wv.most_similar(w, topn=5):
            print(f"   {sim}: {score:.3f}")
    else:
        print(f"\nWord '{w}' not in Word2Vec vocabulary.")


In [ ]:
# Step 5.1: Custom PyTorch Dataset for classification

from torch.utils.data import Dataset

class MentalHealthDataset(Dataset):
    def __init__(self, X, y):
        """
        X: numpy array of shape (num_samples, MAX_LEN)
        y: numpy array of integer class labels
        """
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        input_ids = torch.tensor(self.X[idx], dtype=torch.long)
        label = torch.tensor(self.y[idx], dtype=torch.long)   # CrossEntropy -> long
        
        return input_ids, label


In [ ]:
# Step 5.2: Instantiate PyTorch Dataset objects

train_dataset = MentalHealthDataset(X_train, y_train)
val_dataset   = MentalHealthDataset(X_val, y_val)
test_dataset  = MentalHealthDataset(X_test, y_test)

print("Train samples:", len(train_dataset))
print("Val samples  :", len(val_dataset))
print("Test samples :", len(test_dataset))

In [ ]:
# Step 5.3: Create DataLoaders

from torch.utils.data import DataLoader

BATCH_SIZE = 64  

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False
)

print("Train loader batches:", len(train_loader))
print("Val loader batches  :", len(val_loader))
print("Test loader batches :", len(test_loader))

In [ ]:
# Step 5.4: Inspect shapes from a batch

sample_inputs, sample_labels = next(iter(train_loader))

print("Input batch shape :", sample_inputs.shape)   # (batch_size, MAX_LEN)
print("Labels batch shape:", sample_labels.shape)   # (batch_size)

print("\nExample input_ids:", sample_inputs[0][:20])
print("Example label:", sample_labels[0].item())


In [ ]:
# Step 6.1: Define the LSTM classifier model

import torch.nn as nn
import torch.nn.functional as F

class LSTMClassifier(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        hidden_dim=128,
        num_classes=7,
        dropout=0.3
    ):
        super().__init__()

        vocab_size, embed_dim = embedding_matrix.shape

        # 1) Embedding layer initialized from Word2Vec
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=False   # fine-tune embeddings
        )

        # 2) Single-layer LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        # 3) Dropout
        self.dropout = nn.Dropout(dropout)

        # 4) Final linear layer (7 logits)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)        # (batch, seq_len, embed_dim)
        output, (h_n, c_n) = self.lstm(emb)
        
        # h_n: final hidden state of shape (num_layers, batch, hidden_dim)
        h_last = h_n[-1]               # (batch, hidden_dim)

        out = self.dropout(h_last)
        logits = self.fc(out)

        return logits                  # CrossEntropyLoss expects raw logits


In [ ]:
# Step 6.2: Create model, optimizer, loss function

HIDDEN_DIM = 128
NUM_CLASSES = len(unique_labels)   # = 7
DROPOUT = 0.3

model = LSTMClassifier(
    embedding_matrix=embedding_matrix_tensor,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

In [ ]:
# Step 6.3: Count trainable parameters 

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_parameters(model):,}")


In [ ]:
# Step 7.1: Macro F1 calculation helper

from sklearn.metrics import f1_score

def compute_macro_f1(y_true, y_pred):
    """
    y_true: numpy array (N,)
    y_pred: numpy array (N,)
    """
    return f1_score(y_true, y_pred, average="macro")

In [ ]:
# Step 7.2: Validation loop

def evaluate(model, val_loader, criterion):
    model.eval()

    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits = model(inputs)
            loss = criterion(logits, labels)

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = val_loss / len(val_loader)
    macro_f1 = compute_macro_f1(all_labels, all_preds)

    return avg_loss, macro_f1

In [ ]:
# Step 7.3: Training loop with validation + tracking metrics

EPOCHS = 10

train_losses = []
val_losses = []
val_f1_scores = []

best_val_loss = float("inf")
best_model_path = "best_lstm_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(inputs)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Training loss for this epoch
    avg_train_loss = running_loss / len(train_loader)

    # Validation
    avg_val_loss, macro_f1 = evaluate(model, val_loader, criterion)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    val_f1_scores.append(macro_f1)

    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f}")
    print(f"  Val Macro F1: {macro_f1:.4f}")

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_model_path)
        print("  -> Saved as best model")

print("\nTraining complete.")


In [ ]:
# Step 7.4: Visualize Training vs Validation Loss

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Step 7.5: Visualize Validation Macro F1

plt.figure(figsize=(8, 5))
plt.plot(val_f1_scores, marker='o', color='purple')
plt.xlabel("Epoch")
plt.ylabel("Macro F1 Score")
plt.title("Validation Macro F1 over Epochs")
plt.grid(True)
plt.show()


In [ ]:
# Step 7.6: Load the best model after training

best_model = LSTMClassifier(
    embedding_matrix=embedding_matrix_tensor,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
)

best_model.load_state_dict(torch.load(best_model_path))
best_model = best_model.to(device)

print("Best model loaded.")

In [ ]:
# Step 8.0: Define desired label order and corresponding IDs

LABEL_ORDER = [
    "Normal",
    "Depression",
    "Suicidal",
    "Anxiety",
    "Bipolar",
    "Stress",
    "Personality disorder",
]

# Map label names to their integer IDs (from label2id created earlier)
id_order = [label2id[label] for label in LABEL_ORDER]

print("Label order:", LABEL_ORDER)
print("ID order   :", id_order)

In [ ]:
# Step 8.1: Evaluate best model on the test set

def evaluate_test(model, test_loader):
    model.eval()
    
    all_preds = []
    all_labels = []
    test_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits = model(inputs)
            loss = criterion(logits, labels)

            test_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = test_loss / len(test_loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, macro_f1, all_labels, all_preds


test_loss, test_macro_f1, test_labels, test_preds = evaluate_test(best_model, test_loader)

print("Test Loss:", test_loss)
print("Test Macro F1:", test_macro_f1)

In [ ]:
# Step 8.2: Accuracy & classification report with custom label order

from sklearn.metrics import accuracy_score, classification_report

test_accuracy = accuracy_score(test_labels, test_preds)

print("Test Accuracy:", test_accuracy)
print("\nClassification Report (ordered):")
print(
    classification_report(
        test_labels,
        test_preds,
        labels=id_order,            
        target_names=LABEL_ORDER    
    )
)

In [ ]:
# Step 8.3: Confusion matrix heatmap with custom label order

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(test_labels, test_preds, labels=id_order)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=LABEL_ORDER,
    yticklabels=LABEL_ORDER
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - LSTM + Word2Vec")
plt.tight_layout()
plt.show()


In [ ]:
# Step 8.4: Per-class F1 score bar plot with custom label order & colors

from sklearn.metrics import f1_score

# Compute F1 per class 
f1_per_class = f1_score(
    test_labels,
    test_preds,
    average=None,
    labels=id_order   
)

status_colors = {
    "Normal":               "#8FB9A8", 
    "Depression":           "#F2C57C",  
    "Suicidal":             "#F28F6B", 
    "Anxiety":              "#FFB4A2",  
    "Bipolar":              "#A5B5D9", 
    "Stress":               "#E4B7E5",  
    "Personality disorder": "#A4A3F1"
}

palette = [status_colors[label] for label in LABEL_ORDER]

plt.figure(figsize=(10, 5))
sns.barplot(x=np.array(LABEL_ORDER), y=f1_per_class, palette=palette)
plt.title("Per-class F1 Scores (Test Set)")
plt.ylabel("F1 Score")
plt.xlabel("Class")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

# Print numeric values too
for label, f1val in zip(LABEL_ORDER, f1_per_class):
    print(f"{label:20s}: {f1val:.4f}")


In [ ]:
# Step 9.1: Summary comparison table

import pandas as pd

summary_data = {
    "Model": ["TF-IDF + Linear SVM", "Word2Vec + LSTM"],
    "Macro F1": [0.7118, 0.58],
    "Test Accuracy": [0.7664, 0.7358]
}

summary_df = pd.DataFrame(summary_data)
summary_df


In [ ]:
# Step 9.3: Per-class F1 comparison (Optional)

svm_f1 = [0.9115, 0.7034, 0.6836, 0.7745, 0.7555, 0.5623, 0.5917]
lstm_f1 = [0.93, 0.68, 0.69, 0.66, 0.65, 0.40, 0.03]  # from your output

plt.figure(figsize=(12, 5))
x = range(len(LABEL_ORDER))

plt.plot(x, svm_f1, marker='o', label="SVM")
plt.plot(x, lstm_f1, marker='o', label="LSTM")
plt.xticks(x, LABEL_ORDER, rotation=45, ha="right")
plt.ylabel("F1 Score")
plt.title("Per-Class F1: SVM vs LSTM")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()
